# 🔧 NOC Tune - Phase 1: TTFB Testing

**Network Operations Center Tuning & Diagnostic Tool**

Tools untuk mengukur dan menganalisis Time to First Byte (TTFB) dan DNS resolution performance.

## Fitur Notebook Ini:
- ✅ Menjalankan `dig` command dengan berbagai DNS server dan domain
- ✅ Mengukur TTFB menggunakan `curl` dengan breakdown timing lengkap
- ✅ Multi-sample testing dengan delay yang dapat dikonfigurasi
- ✅ Export hasil ke CSV dengan timestamp otomatis
- ✅ Visualisasi perbandingan performa

## Threshold OpenSignal (BCQ)
| Metric | Target |
|--------|--------|
| TTFB | < 800ms |
| Latency | < 50ms |
| Jitter | < 12ms |

---
**Referensi:** TTFB Troubleshooting Guide - Telkom Indonesia/Infranexia

## 1. Import Libraries & Setup Environment

In [ ]:
# Import required libraries
import subprocess
import pandas as pd
import numpy as np
from datetime import datetime
import time
import os
import re
import platform
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Create results directory if not exists
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

print("✅ Libraries imported successfully!")
print(f"📁 Results will be saved to: {RESULTS_DIR.absolute()}")

## 2. Configuration Parameters

Sesuaikan parameter di bawah ini sebelum menjalankan test:
- `SAMPLE_COUNT`: Jumlah pengulangan test (default: 5)
- `DELAY_SECONDS`: Jeda antar test dalam detik (default: 5)
- `DNS_SERVERS`: Daftar DNS server yang akan ditest
- `TARGET_DOMAINS`: Daftar domain target

In [ ]:
# ============================================
# CONFIGURATION - Sesuaikan di sini!
# ============================================

# Testing parameters
SAMPLE_COUNT = 5        # Jumlah pengulangan test
DELAY_SECONDS = 5       # Jeda antar test (detik)

# DNS Servers to test
DNS_SERVERS = {
    "Google DNS Primary": "8.8.8.8",
    "Google DNS Secondary": "8.8.4.4",
    "Cloudflare DNS": "1.1.1.1",
    # ISP DNS akan ditambahkan otomatis
}

# Target domains for testing
TARGET_DOMAINS = [
    "www.instagram.com",
    "qt-google-cloud-cdn.bronze.systems",
    # Tambahkan domain lain jika perlu:
    # "www.google.com",
    # "www.youtube.com",
]

# TTFB Thresholds (ms) - based on OpenSignal BCQ
THRESHOLDS = {
    "good": 600,      # < 600ms = Good
    "warning": 800,   # 600-800ms = Needs Improvement
    "poor": 800       # > 800ms = Poor
}

print("📋 Configuration loaded:")
print(f"   • Sample count: {SAMPLE_COUNT}")
print(f"   • Delay: {DELAY_SECONDS}s")
print(f"   • DNS Servers: {list(DNS_SERVERS.keys())}")
print(f"   • Target Domains: {TARGET_DOMAINS}")

## 3. DNS Server Detection Functions

Fungsi untuk mendeteksi DNS server dari ISP (IndiHome/Telkom) secara otomatis.

In [ ]:
def detect_isp_dns():
    """
    Detect ISP DNS servers from system configuration.
    Works on macOS and Linux.
    """
    isp_dns = []
    system = platform.system()
    
    try:
        if system == "Darwin":  # macOS
            # Use scutil to get DNS configuration
            result = subprocess.run(
                ["scutil", "--dns"],
                capture_output=True, text=True, timeout=10
            )
            if result.returncode == 0:
                # Parse nameserver entries
                for line in result.stdout.split('\n'):
                    if 'nameserver[' in line:
                        match = re.search(r':\s*(\d+\.\d+\.\d+\.\d+)', line)
                        if match:
                            ip = match.group(1)
                            # Exclude common public DNS
                            if ip not in ['8.8.8.8', '8.8.4.4', '1.1.1.1', '1.0.0.1']:
                                if ip not in isp_dns:
                                    isp_dns.append(ip)
        
        elif system == "Linux":
            # Read from /etc/resolv.conf
            with open('/etc/resolv.conf', 'r') as f:
                for line in f:
                    if line.startswith('nameserver'):
                        ip = line.split()[1].strip()
                        if ip not in ['8.8.8.8', '8.8.4.4', '1.1.1.1', '1.0.0.1']:
                            if ip not in isp_dns:
                                isp_dns.append(ip)
        
        # Also try using systemd-resolve if available (Linux)
        if system == "Linux":
            try:
                result = subprocess.run(
                    ["systemd-resolve", "--status"],
                    capture_output=True, text=True, timeout=10
                )
                if result.returncode == 0:
                    for line in result.stdout.split('\n'):
                        if 'DNS Servers:' in line:
                            parts = line.split(':')
                            if len(parts) > 1:
                                ip = parts[1].strip()
                                if ip and ip not in isp_dns:
                                    isp_dns.append(ip)
            except:
                pass
                
    except Exception as e:
        print(f"⚠️ Error detecting ISP DNS: {e}")
    
    return isp_dns

# Detect and add ISP DNS
detected_isp_dns = detect_isp_dns()

if detected_isp_dns:
    print(f"🔍 Detected ISP DNS servers: {detected_isp_dns}")
    for i, dns in enumerate(detected_isp_dns[:2]):  # Limit to first 2
        DNS_SERVERS[f"ISP DNS {i+1}"] = dns
else:
    print("⚠️ Could not detect ISP DNS servers automatically")
    print("   You can manually add ISP DNS in the configuration cell above")

print(f"\n📋 Final DNS servers list:")
for name, ip in DNS_SERVERS.items():
    print(f"   • {name}: {ip}")

## 4. TTFB Measurement Functions Using Curl

Mengukur TTFB breakdown menggunakan `curl`:
- `time_namelookup`: DNS resolution time
- `time_connect`: TCP 3-way handshake
- `time_appconnect`: SSL/TLS negotiation
- `time_starttransfer`: **TTFB** - waktu hingga byte pertama
- `time_total`: Total transfer time

In [ ]:
def measure_ttfb_curl(url: str, dns_server: str = None) -> dict:
    """
    Measure TTFB using curl with detailed timing breakdown.
    
    Args:
        url: Target URL (e.g., https://www.instagram.com)
        dns_server: Optional DNS server to use
        
    Returns:
        Dictionary with timing metrics in milliseconds
    """
    timestamp = datetime.now().isoformat()
    
    # Build curl command with timing format
    curl_format = (
        '{"time_namelookup": %{time_namelookup}, '
        '"time_connect": %{time_connect}, '
        '"time_appconnect": %{time_appconnect}, '
        '"time_starttransfer": %{time_starttransfer}, '
        '"time_total": %{time_total}, '
        '"http_code": %{http_code}}'
    )
    
    cmd = [
        "curl",
        "-o", "/dev/null",  # Discard output
        "-s",               # Silent mode
        "-w", curl_format,  # Format output
        "--max-time", "30", # Timeout 30s
    ]
    
    # Add DNS server if specified (requires curl 7.21.3+)
    if dns_server:
        cmd.extend(["--dns-servers", dns_server])
    
    # Ensure URL has protocol
    if not url.startswith('http'):
        url = f"https://{url}"
    
    cmd.append(url)
    
    result = {
        "timestamp": timestamp,
        "url": url,
        "dns_server": dns_server or "system_default",
        "lookup_ms": None,
        "connect_ms": None,
        "appconnect_ms": None,
        "ttfb_ms": None,
        "total_ms": None,
        "tcp_rtt_ms": None,
        "tls_ms": None,
        "server_response_ms": None,
        "http_code": None,
        "error": None,
        "status": "unknown"
    }
    
    try:
        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=35
        )
        
        if proc.returncode == 0:
            import json
            data = json.loads(proc.stdout)
            
            # Convert to milliseconds
            result["lookup_ms"] = round(data["time_namelookup"] * 1000, 2)
            result["connect_ms"] = round(data["time_connect"] * 1000, 2)
            result["appconnect_ms"] = round(data["time_appconnect"] * 1000, 2)
            result["ttfb_ms"] = round(data["time_starttransfer"] * 1000, 2)
            result["total_ms"] = round(data["time_total"] * 1000, 2)
            result["http_code"] = data["http_code"]
            
            # Calculate derived metrics
            result["tcp_rtt_ms"] = round(result["connect_ms"] - result["lookup_ms"], 2)
            result["tls_ms"] = round(result["appconnect_ms"] - result["connect_ms"], 2)
            result["server_response_ms"] = round(result["ttfb_ms"] - result["appconnect_ms"], 2)
            
            # Determine status based on TTFB threshold
            if result["ttfb_ms"] < THRESHOLDS["good"]:
                result["status"] = "good"
            elif result["ttfb_ms"] < THRESHOLDS["warning"]:
                result["status"] = "warning"
            else:
                result["status"] = "poor"
                
        else:
            result["error"] = proc.stderr.strip() or f"Exit code: {proc.returncode}"
            result["status"] = "error"
            
    except subprocess.TimeoutExpired:
        result["error"] = "Timeout (>30s)"
        result["status"] = "timeout"
    except Exception as e:
        result["error"] = str(e)
        result["status"] = "error"
    
    return result

# Test the function
print("🧪 Testing curl TTFB measurement...")
test_result = measure_ttfb_curl("https://www.google.com")
if test_result["error"]:
    print(f"⚠️ Test result: {test_result['error']}")
    print("   Note: --dns-servers flag requires curl 7.21.3+ and may not work on all systems")
else:
    print(f"✅ Test successful! TTFB: {test_result['ttfb_ms']}ms ({test_result['status']})")

## 5. Dig Command Execution Functions

Fungsi untuk menjalankan `dig @{dns_server} {domain} +trace` dan mengekstrak query time.

In [ ]:
def run_dig_trace(domain: str, dns_server: str) -> dict:
    """
    Execute 'dig @{dns_server} {domain} +trace' command.
    
    Args:
        domain: Target domain (e.g., www.instagram.com)
        dns_server: DNS server IP address
        
    Returns:
        Dictionary with dig results and timing
    """
    timestamp = datetime.now().isoformat()
    
    # Build dig command
    cmd = ["dig", f"@{dns_server}", domain, "+trace", "+stats"]
    
    result = {
        "timestamp": timestamp,
        "domain": domain,
        "dns_server": dns_server,
        "query_time_ms": None,
        "total_queries": 0,
        "final_answer": None,
        "trace_hops": [],
        "raw_output": None,
        "error": None,
        "status": "unknown"
    }
    
    try:
        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=30
        )
        
        result["raw_output"] = proc.stdout
        
        if proc.returncode == 0:
            output = proc.stdout
            
            # Extract query times
            query_times = []
            for line in output.split('\n'):
                # Look for query time lines
                if 'Query time:' in line:
                    match = re.search(r'Query time:\s*(\d+)\s*msec', line)
                    if match:
                        query_times.append(int(match.group(1)))
                
                # Extract final A record answer
                if '\tA\t' in line and not line.strip().startswith(';'):
                    parts = line.split()
                    if len(parts) >= 5:
                        result["final_answer"] = parts[-1]
            
            result["total_queries"] = len(query_times)
            result["query_time_ms"] = sum(query_times) if query_times else None
            result["trace_hops"] = query_times
            
            # Determine status
            if result["query_time_ms"] is not None:
                if result["query_time_ms"] < 100:
                    result["status"] = "good"
                elif result["query_time_ms"] < 200:
                    result["status"] = "warning"
                else:
                    result["status"] = "poor"
            else:
                result["status"] = "no_timing"
                
        else:
            result["error"] = proc.stderr.strip() or f"Exit code: {proc.returncode}"
            result["status"] = "error"
            
    except subprocess.TimeoutExpired:
        result["error"] = "Timeout (>30s)"
        result["status"] = "timeout"
    except FileNotFoundError:
        result["error"] = "dig command not found. Install dnsutils/bind-utils."
        result["status"] = "error"
    except Exception as e:
        result["error"] = str(e)
        result["status"] = "error"
    
    return result

def run_dig_simple(domain: str, dns_server: str) -> dict:
    """
    Execute simple 'dig @{dns_server} {domain}' without trace.
    Faster for basic DNS lookup timing.
    """
    timestamp = datetime.now().isoformat()
    cmd = ["dig", f"@{dns_server}", domain, "+stats"]
    
    result = {
        "timestamp": timestamp,
        "domain": domain,
        "dns_server": dns_server,
        "query_time_ms": None,
        "resolved_ip": None,
        "error": None,
        "status": "unknown"
    }
    
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        
        if proc.returncode == 0:
            output = proc.stdout
            
            # Extract query time
            match = re.search(r'Query time:\s*(\d+)\s*msec', output)
            if match:
                result["query_time_ms"] = int(match.group(1))
            
            # Extract resolved IP
            for line in output.split('\n'):
                if '\tA\t' in line and not line.strip().startswith(';'):
                    parts = line.split()
                    if len(parts) >= 5:
                        result["resolved_ip"] = parts[-1]
                        break
            
            result["status"] = "success"
        else:
            result["error"] = proc.stderr.strip()
            result["status"] = "error"
            
    except Exception as e:
        result["error"] = str(e)
        result["status"] = "error"
    
    return result

# Test dig function
print("🧪 Testing dig trace function...")
test_dig = run_dig_trace("www.google.com", "8.8.8.8")
if test_dig["error"]:
    print(f"⚠️ Dig test error: {test_dig['error']}")
else:
    print(f"✅ Dig test successful!")
    print(f"   Total query time: {test_dig['query_time_ms']}ms")
    print(f"   Trace hops: {test_dig['total_queries']}")

## 6. Batch Test Runner with Delay Control

Fungsi untuk menjalankan test secara batch dengan delay yang dapat dikonfigurasi.

In [ ]:
def batch_run_dig_tests(
    domains: list,
    dns_servers: dict,
    sample_count: int = 5,
    delay_seconds: int = 5,
    use_trace: bool = True
) -> list:
    """
    Run batch dig tests for multiple domains and DNS servers.
    
    Args:
        domains: List of target domains
        dns_servers: Dict of {name: ip} for DNS servers
        sample_count: Number of samples per combination
        delay_seconds: Delay between tests
        use_trace: Whether to use +trace flag
        
    Returns:
        List of test results
    """
    results = []
    total_tests = len(domains) * len(dns_servers) * sample_count
    
    print(f"🚀 Starting batch dig tests...")
    print(f"   Domains: {len(domains)}")
    print(f"   DNS Servers: {len(dns_servers)}")
    print(f"   Samples per combination: {sample_count}")
    print(f"   Total tests: {total_tests}")
    print(f"   Estimated time: ~{total_tests * (delay_seconds + 2) // 60} minutes")
    print("-" * 50)
    
    test_func = run_dig_trace if use_trace else run_dig_simple
    
    with tqdm(total=total_tests, desc="Running tests") as pbar:
        for domain in domains:
            for dns_name, dns_ip in dns_servers.items():
                for sample_num in range(1, sample_count + 1):
                    # Run test
                    result = test_func(domain, dns_ip)
                    result["dns_name"] = dns_name
                    result["sample_num"] = sample_num
                    results.append(result)
                    
                    # Update progress
                    status_icon = "✅" if result["status"] == "good" else "⚠️" if result["status"] == "warning" else "❌"
                    pbar.set_postfix({
                        "domain": domain[:20],
                        "dns": dns_name[:15],
                        "time": f"{result['query_time_ms']}ms" if result['query_time_ms'] else "N/A"
                    })
                    pbar.update(1)
                    
                    # Delay before next test (except for last one)
                    if not (domain == domains[-1] and 
                            dns_name == list(dns_servers.keys())[-1] and 
                            sample_num == sample_count):
                        time.sleep(delay_seconds)
    
    print("-" * 50)
    print(f"✅ Completed {len(results)} tests")
    
    return results


def batch_run_ttfb_tests(
    domains: list,
    sample_count: int = 5,
    delay_seconds: int = 5
) -> list:
    """
    Run batch TTFB tests using curl for multiple domains.
    
    Args:
        domains: List of target domains/URLs
        sample_count: Number of samples per domain
        delay_seconds: Delay between tests
        
    Returns:
        List of TTFB test results
    """
    results = []
    total_tests = len(domains) * sample_count
    
    print(f"🚀 Starting batch TTFB tests...")
    print(f"   Domains: {len(domains)}")
    print(f"   Samples per domain: {sample_count}")
    print(f"   Total tests: {total_tests}")
    print(f"   Delay between tests: {delay_seconds}s")
    print("-" * 50)
    
    with tqdm(total=total_tests, desc="Measuring TTFB") as pbar:
        for domain in domains:
            for sample_num in range(1, sample_count + 1):
                # Ensure https protocol
                url = domain if domain.startswith('http') else f"https://{domain}"
                
                # Run TTFB test
                result = measure_ttfb_curl(url)
                result["sample_num"] = sample_num
                results.append(result)
                
                # Update progress
                pbar.set_postfix({
                    "domain": domain[:25],
                    "TTFB": f"{result['ttfb_ms']}ms" if result['ttfb_ms'] else "ERR"
                })
                pbar.update(1)
                
                # Delay before next test
                if not (domain == domains[-1] and sample_num == sample_count):
                    time.sleep(delay_seconds)
    
    print("-" * 50)
    print(f"✅ Completed {len(results)} TTFB tests")
    
    return results

print("✅ Batch test functions defined")

## 7. CSV Export with Timestamped Filenames

Fungsi untuk menyimpan hasil ke CSV dengan format penamaan yang jelas.

In [ ]:
def export_to_csv(results: list, test_type: str = "dig") -> str:
    """
    Export test results to CSV with timestamped filename.
    
    Args:
        results: List of test result dictionaries
        test_type: Type of test ("dig" or "ttfb")
        
    Returns:
        Path to saved CSV file
    """
    if not results:
        print("⚠️ No results to export")
        return None
    
    # Generate filename with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"noctune_{test_type}_{timestamp}.csv"
    filepath = RESULTS_DIR / filename
    
    # Convert to DataFrame
    df = pd.DataFrame(results)
    
    # Remove raw_output column if exists (too large for CSV)
    if 'raw_output' in df.columns:
        df = df.drop(columns=['raw_output'])
    
    # Save to CSV
    df.to_csv(filepath, index=False)
    
    print(f"💾 Results saved to: {filepath}")
    print(f"   Total records: {len(df)}")
    
    return str(filepath)


def load_results_csv(filepath: str) -> pd.DataFrame:
    """Load results from CSV file."""
    return pd.read_csv(filepath)


def list_result_files() -> list:
    """List all CSV result files in the results directory."""
    files = sorted(RESULTS_DIR.glob("noctune_*.csv"), reverse=True)
    
    if not files:
        print("📁 No result files found")
        return []
    
    print(f"📁 Found {len(files)} result file(s):")
    for f in files:
        # Get file size
        size_kb = f.stat().st_size / 1024
        print(f"   • {f.name} ({size_kb:.1f} KB)")
    
    return [str(f) for f in files]

print("✅ CSV export functions defined")
print(f"📁 Results directory: {RESULTS_DIR.absolute()}")

## 8. TTFB Data Visualization

Visualisasi hasil pengujian TTFB dengan breakdown komponen timing.

In [ ]:
def plot_ttfb_breakdown(df: pd.DataFrame, title: str = "TTFB Breakdown"):
    """
    Plot TTFB breakdown by components (DNS, TCP, TLS, Server Response).
    """
    # Filter out errors
    valid_df = df[df['ttfb_ms'].notna()].copy()
    
    if valid_df.empty:
        print("⚠️ No valid data to plot")
        return
    
    # Group by URL and calculate means
    if 'url' in valid_df.columns:
        grouped = valid_df.groupby('url').agg({
            'lookup_ms': 'mean',
            'tcp_rtt_ms': 'mean',
            'tls_ms': 'mean',
            'server_response_ms': 'mean',
            'ttfb_ms': 'mean'
        }).reset_index()
    else:
        grouped = valid_df[['lookup_ms', 'tcp_rtt_ms', 'tls_ms', 'server_response_ms', 'ttfb_ms']].mean().to_frame().T
        grouped['url'] = 'All'
    
    # Create stacked bar chart
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = range(len(grouped))
    width = 0.6
    
    # Stack the components
    bottom = np.zeros(len(grouped))
    
    components = [
        ('lookup_ms', 'DNS Lookup', '#2196F3'),
        ('tcp_rtt_ms', 'TCP RTT', '#FF9800'),
        ('tls_ms', 'TLS Handshake', '#4CAF50'),
        ('server_response_ms', 'Server Response', '#F44336')
    ]
    
    for col, label, color in components:
        values = grouped[col].fillna(0).values
        ax.bar(x, values, width, bottom=bottom, label=label, color=color)
        bottom += values
    
    # Add TTFB values on top
    for i, (idx, row) in enumerate(grouped.iterrows()):
        ax.text(i, row['ttfb_ms'] + 20, f"{row['ttfb_ms']:.0f}ms", 
                ha='center', va='bottom', fontweight='bold')
    
    # Add threshold lines
    ax.axhline(y=THRESHOLDS['good'], color='green', linestyle='--', alpha=0.7, label=f"Good (<{THRESHOLDS['good']}ms)")
    ax.axhline(y=THRESHOLDS['warning'], color='red', linestyle='--', alpha=0.7, label=f"Poor (>{THRESHOLDS['warning']}ms)")
    
    ax.set_xlabel('Endpoint')
    ax.set_ylabel('Time (ms)')
    ax.set_title(title)
    ax.set_xticks(x)
    
    # Shorten URL labels
    labels = [url.replace('https://', '').replace('http://', '')[:30] for url in grouped['url']]
    ax.set_xticklabels(labels, rotation=45, ha='right')
    
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()
    
    return fig


def plot_dig_comparison(df: pd.DataFrame, title: str = "DNS Query Time Comparison"):
    """
    Plot DNS query time comparison across different DNS servers and domains.
    """
    valid_df = df[df['query_time_ms'].notna()].copy()
    
    if valid_df.empty:
        print("⚠️ No valid data to plot")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Box plot by DNS server
    if 'dns_name' in valid_df.columns:
        ax1 = axes[0]
        valid_df.boxplot(column='query_time_ms', by='dns_name', ax=ax1)
        ax1.set_title('Query Time by DNS Server')
        ax1.set_xlabel('DNS Server')
        ax1.set_ylabel('Query Time (ms)')
        plt.sca(ax1)
        plt.xticks(rotation=45, ha='right')
    
    # Plot 2: Box plot by domain
    if 'domain' in valid_df.columns:
        ax2 = axes[1]
        valid_df.boxplot(column='query_time_ms', by='domain', ax=ax2)
        ax2.set_title('Query Time by Domain')
        ax2.set_xlabel('Domain')
        ax2.set_ylabel('Query Time (ms)')
        plt.sca(ax2)
        plt.xticks(rotation=45, ha='right')
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()
    
    return fig


def print_summary_stats(df: pd.DataFrame, metric_col: str = 'ttfb_ms'):
    """Print summary statistics for results."""
    valid_df = df[df[metric_col].notna()]
    
    if valid_df.empty:
        print("⚠️ No valid data for summary")
        return
    
    print("=" * 60)
    print("📊 SUMMARY STATISTICS")
    print("=" * 60)
    
    print(f"\n📈 {metric_col.upper()} Statistics:")
    print(f"   • Count:  {len(valid_df)}")
    print(f"   • Mean:   {valid_df[metric_col].mean():.2f} ms")
    print(f"   • Median: {valid_df[metric_col].median():.2f} ms")
    print(f"   • Min:    {valid_df[metric_col].min():.2f} ms")
    print(f"   • Max:    {valid_df[metric_col].max():.2f} ms")
    print(f"   • Std:    {valid_df[metric_col].std():.2f} ms")
    
    # Status distribution
    if 'status' in valid_df.columns:
        print(f"\n📊 Status Distribution:")
        for status, count in valid_df['status'].value_counts().items():
            pct = count / len(valid_df) * 100
            icon = "✅" if status == "good" else "⚠️" if status == "warning" else "❌"
            print(f"   {icon} {status}: {count} ({pct:.1f}%)")
    
    print("=" * 60)

print("✅ Visualization functions defined")

---

## 9. 🚀 Interactive Test Execution

Jalankan cell di bawah untuk memulai pengujian. Hasil akan otomatis disimpan ke CSV.

### 9.1 Run DIG Trace Tests

Menjalankan `dig @{dns_server} {domain} +trace` untuk semua kombinasi DNS server dan domain yang dikonfigurasi.

In [ ]:
# =============================================
# 🔄 RUN DIG TRACE TESTS
# Klik "Run" untuk menjalankan semua test dig
# =============================================

# Run batch dig tests
dig_results = batch_run_dig_tests(
    domains=TARGET_DOMAINS,
    dns_servers=DNS_SERVERS,
    sample_count=SAMPLE_COUNT,
    delay_seconds=DELAY_SECONDS,
    use_trace=True
)

# Export to CSV
dig_csv_path = export_to_csv(dig_results, test_type="dig_trace")

# Convert to DataFrame for analysis
dig_df = pd.DataFrame(dig_results)

# Show summary
print_summary_stats(dig_df, metric_col='query_time_ms')

In [ ]:
# Visualize dig results
if 'dig_df' in dir() and not dig_df.empty:
    plot_dig_comparison(dig_df, title="DNS Query Time Analysis")
    
    # Show data preview
    print("\n📋 Data Preview:")
    display(dig_df[['timestamp', 'domain', 'dns_name', 'query_time_ms', 'status']].head(10))

### 9.2 Run TTFB Tests (curl)

Menjalankan pengukuran TTFB menggunakan curl dengan breakdown timing lengkap.

In [ ]:
# =============================================
# 🔄 RUN TTFB TESTS (CURL)
# Klik "Run" untuk menjalankan semua test TTFB  
# =============================================

# Run batch TTFB tests
ttfb_results = batch_run_ttfb_tests(
    domains=TARGET_DOMAINS,
    sample_count=SAMPLE_COUNT,
    delay_seconds=DELAY_SECONDS
)

# Export to CSV
ttfb_csv_path = export_to_csv(ttfb_results, test_type="ttfb_curl")

# Convert to DataFrame for analysis
ttfb_df = pd.DataFrame(ttfb_results)

# Show summary
print_summary_stats(ttfb_df, metric_col='ttfb_ms')

In [ ]:
# Visualize TTFB results
if 'ttfb_df' in dir() and not ttfb_df.empty:
    plot_ttfb_breakdown(ttfb_df, title="TTFB Breakdown Analysis")
    
    # Show detailed data preview
    print("\n📋 TTFB Data Preview:")
    display(ttfb_df[['timestamp', 'url', 'lookup_ms', 'connect_ms', 'appconnect_ms', 
                     'ttfb_ms', 'server_response_ms', 'status']].head(10))

---

## 10. 📁 Hasil & Utility

### List Result Files

In [ ]:
# List all saved result files
result_files = list_result_files()

### Load Previous Results

Untuk memuat hasil pengujian sebelumnya, isi path file CSV di bawah:

In [ ]:
# Load previous results (uncomment and edit path)
# ================================================

# csv_path = "./results/noctune_ttfb_curl_YYYYMMDD_HHMMSS.csv"
# loaded_df = load_results_csv(csv_path)
# print_summary_stats(loaded_df, metric_col='ttfb_ms')
# plot_ttfb_breakdown(loaded_df)